# D3 · Anscombe: four datasets, one line, three wrong models

**Type**: Interpret · **Level**: ●●○ independent · **Time**: about 40 minutes · **core path ★**

## Objective

Verify that the four datasets of Anscombe's quartet have the same summary statistics, and then use leverage and
Cook's distance to understand which point moves the line in each one.

## Dataset

`ds.anscombe()`: four datasets of 11 points (`"I"`, `"II"`, `"III"`, `"IV"`), with columns `dataset`, `x`, `y`. Built in 1973
to have almost identical slope, intercept, R² and residual variance, even though they are very different.

## Educational purpose

Understand **why** summary statistics are not enough and learn the two measures that expose the points that
command the model. Dataset IV contains an instructive limit case: a point with leverage exactly equal to 1.

## Task

1. For each of the four datasets do the OLS fit and build `summary` (index `I`..`IV`, columns `slope`, `intercept`, `r2`, `resid_std`, the latter with `ddof=2`).
2. Write `leverages(x)` (diagonal of the hat matrix with intercept) and `cook(x, y)` (Cook's distance of each point, with the formula of lesson 1.3). If the leverage of a point exceeds `1 - 1e-9`, the formula gives 0/0: assign `np.inf` to that point.
3. Assign to `influential_point` a dictionary `{name: index_of_the_point}` with the point at maximum Cook's distance in every dataset.
4. In dataset III remove the influential point and recompute `slope_III_without` and `r2_III_without`. In IV remove the influential point and assign to `iv_x_constant` `True` if all the remaining x are equal.
5. Assign to `regression_suitable` a dictionary `{name: True/False}`: for which datasets is an OLS line a good model **without interventions**?

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LinearRegression

anscombe = ds.anscombe()
sets = {name: g[["x", "y"]].reset_index(drop=True) for name, g in anscombe.groupby("dataset")}
sets["I"].head(3)

## Your work

In [ ]:
rows = {}
for name, d in sets.items():
    ...
summary = pd.DataFrame(rows).T
summary.round(3)

In [ ]:
def leverages(x):
    ...

def cook(x, y):
    ...

In [ ]:
influential_point = {}
for name, d in sets.items():
    ...
influential_point

In [ ]:
slope_III_without = ...
r2_III_without = ...
iv_x_constant = ...

In [ ]:
regression_suitable = {"I": ..., "II": ..., "III": ..., "IV": ...}

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9, 6.5), sharex=True, sharey=True)
# one panel per dataset: points, OLS line and the influential point highlighted

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='Cook: `e = y - A @ beta`, `s2 = e @ e / (n - p)`, `D = e**2 / (p * s2) * h / (1 - h)**2`; then `D = np.where(h > 1 - 1e-9, np.inf, D)`. To avoid division-by-zero warnings, use `with np.errstate(all="ignore")`.'):
    check(list(summary.columns) == ["slope", "intercept", "r2", "resid_std"],
          "the summary has the required columns", "columns: slope, intercept, r2, resid_std")
    check(np.allclose(summary.slope, 0.5, atol=0.01) and np.allclose(summary.intercept, 3.0, atol=0.01),
          "slope and intercept are practically identical in the four datasets (0.5 and 3.0)",
          "the four lines should have slope 0.5 and intercept 3.0")
    check(np.allclose(summary.r2, 0.667, atol=0.01), "the R2 is identical (0.667) in the four datasets",
          "the R2 should be about 0.667 in all four")
    check(abs(leverages(sets["IV"].x.to_numpy(float)).max() - 1) < 1e-9, "in dataset IV one point has leverage 1",
          "the point with x = 19 is the only one with that value: its leverage is 1")
    check(np.isinf(cook(sets["IV"].x.to_numpy(float), sets["IV"].y.to_numpy(float))).sum() == 1,
          "the Cook's distance of that point is infinite", "assign np.inf to the points with leverage > 1 - 1e-9")
    check(influential_point["III"] == 2 and influential_point["IV"] == 7,
          "influential points: index 2 in III (the vertical outlier), index 7 in IV (the isolated point)",
          "in III the outlier has x = 13, y = 12.74 (index 2); in IV the point with x = 19 has index 7")
    check(abs(slope_III_without - 0.345) < 0.01 and r2_III_without > 0.99,
          "without the outlier dataset III is almost perfectly linear (slope 0.345, R2 ~ 1)",
          "remove the point with index 2 from dataset III and refit")
    check(bool(iv_x_constant), "in dataset IV, without that point, x is constant and the slope is not identifiable",
          "after removing the point x = 19 all the x equal 8")
    check(regression_suitable == {"I": True, "II": False, "III": False, "IV": False}, "correct judgment on the four datasets",
          "only dataset I is a normal linear cloud: II is curved, III has an outlier, IV depends on one point")
done()

---
The commented solution is in `solutions/D_interpret/D3_anscombe_leverage_and_influence.ipynb`. Open it after you pass the check, or after you have really tried.